# 01 - Analise Exploratoria de Dados (EDA)

Analise completa dos dados de consumo energetico de Portugal.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('Bibliotecas carregadas')

## 1. Carregamento dos Dados

In [ ]:
df = pd.read_parquet('../data/processed/processed_data.parquet')

print(f"Dimensoes: {df.shape}")
print(f"Periodo: {df['timestamp'].min()} a {df['timestamp'].max()}")
print(f"Regioes: {df['region'].nunique()} - {sorted(df['region'].unique())}")
print(f"\nColunas ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col:25s} {str(df[col].dtype):10s} nulls={df[col].isnull().sum()}")

## 2. Estatisticas Descritivas

In [ ]:
target = 'consumption_mw'

print("=" * 70)
print("ESTATISTICAS DO CONSUMO ENERGETICO (MW)")
print("=" * 70)
desc = df[target].describe()
for k, v in desc.items():
    print(f"  {k:10s}: {v:>12.2f}")

print(f"\n  Skewness : {df[target].skew():>12.4f}")
print(f"  Kurtosis : {df[target].kurtosis():>12.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df[target], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df[target].mean(), color='red', linestyle='--', label=f'Media={df[target].mean():.0f}')
axes[0].axvline(df[target].median(), color='orange', linestyle='--', label=f'Mediana={df[target].median():.0f}')
axes[0].set_xlabel('Consumo (MW)')
axes[0].set_ylabel('Frequencia')
axes[0].set_title('Distribuicao do Consumo')
axes[0].legend()

stats.probplot(df[target], dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')

df.boxplot(column=target, by='region', ax=axes[2])
axes[2].set_title('Consumo por Regiao')
axes[2].set_xlabel('Regiao')
axes[2].set_ylabel('Consumo (MW)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 3. Padroes Temporais

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

hourly = df.groupby('hour')[target].agg(['mean', 'std'])
axes[0,0].plot(hourly.index, hourly['mean'], 'b-o', linewidth=2)
axes[0,0].fill_between(hourly.index, hourly['mean'] - hourly['std'], hourly['mean'] + hourly['std'], alpha=0.2)
axes[0,0].set_xlabel('Hora')
axes[0,0].set_ylabel('Consumo Medio (MW)')
axes[0,0].set_title('Perfil Horario')
axes[0,0].set_xticks(range(0, 24))

days = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
daily = df.groupby('day_of_week')[target].mean()
colors = ['steelblue']*5 + ['coral']*2
axes[0,1].bar(range(7), daily.values, color=colors, edgecolor='black')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(days)
axes[0,1].set_ylabel('Consumo Medio (MW)')
axes[0,1].set_title('Perfil Semanal')

monthly = df.groupby('month')[target].mean()
axes[1,0].bar(range(1,13), monthly.values, color='steelblue', edgecolor='black')
axes[1,0].set_xticks(range(1,13))
axes[1,0].set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'])
axes[1,0].set_ylabel('Consumo Medio (MW)')
axes[1,0].set_title('Perfil Mensal')

pivot = df.groupby(['day_of_week', 'hour'])[target].mean().unstack()
sns.heatmap(pivot, cmap='YlOrRd', ax=axes[1,1], xticklabels=2)
axes[1,1].set_yticklabels(days, rotation=0)
axes[1,1].set_xlabel('Hora')
axes[1,1].set_title('Mapa de Calor: Hora x Dia da Semana')

plt.tight_layout()
plt.show()

print(f"Hora de pico: {hourly['mean'].idxmax()}h ({hourly['mean'].max():.0f} MW)")
print(f"Hora de vale: {hourly['mean'].idxmin()}h ({hourly['mean'].min():.0f} MW)")

## 4. Analise Regional

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

regional = df.groupby('region')[target].agg(['mean', 'std', 'min', 'max']).sort_values('mean', ascending=True)
regional['mean'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Consumo Medio (MW)')
axes[0].set_title('Consumo Medio por Regiao')

for region in sorted(df['region'].unique()):
    hourly_r = df[df['region'] == region].groupby('hour')[target].mean()
    axes[1].plot(hourly_r.index, hourly_r.values, '-o', label=region, linewidth=2, markersize=4)
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Consumo (MW)')
axes[1].set_title('Perfil Horario por Regiao')
axes[1].legend()
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

print("\nEstatisticas por Regiao:")
print(regional.to_string())

## 5. Correlacoes Meteorologicas

In [ ]:
weather_cols = [c for c in ['temperature', 'humidity', 'wind_speed', 'precipitation', 'cloud_cover', 'pressure'] if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(weather_cols):
    ax = axes[i // 3, i % 3]
    sample = df.sample(min(5000, len(df)), random_state=42)
    ax.scatter(sample[col], sample[target], alpha=0.3, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Consumo (MW)')
    corr = df[col].corr(df[target])
    ax.set_title(f'{col} (r={corr:.4f})')

plt.suptitle('Correlacoes Meteorologicas', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\nCorrelacoes com consumo:")
for col in weather_cols:
    r = df[col].corr(df[target])
    print(f"  {col:20s}: {r:+.4f}")

## 6. Analise de Autocorrelacao

In [ ]:
df_lisboa = df[df['region'] == 'Lisboa'].sort_values('timestamp').set_index('timestamp')

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(df_lisboa[target].dropna().values[:2000], lags=72, ax=axes[0])
axes[0].set_title('ACF - Autocorrelacao (Lisboa)')
axes[0].set_xlabel('Lag (horas)')

plot_pacf(df_lisboa[target].dropna().values[:2000], lags=72, ax=axes[1], method='ywm')
axes[1].set_title('PACF - Autocorrelacao Parcial (Lisboa)')
axes[1].set_xlabel('Lag (horas)')

plt.tight_layout()
plt.show()

print("Observacoes:")
print("  - Forte autocorrelacao com periodicidade de 24h (padrao diario)")
print("  - Decaimento lento da ACF indica tendencia ou sazonalidade longa")

## 7. Decomposicao Sazonal

In [ ]:
df_decomp = df_lisboa[target].resample('h').mean().dropna()[:90*24]

decomposition = seasonal_decompose(df_decomp, model='additive', period=24)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
decomposition.observed.plot(ax=axes[0], title='Observado')
decomposition.trend.plot(ax=axes[1], title='Tendencia')
decomposition.seasonal.plot(ax=axes[2], title='Sazonalidade (24h)')
decomposition.resid.plot(ax=axes[3], title='Residuo')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

var_total = df_decomp.var()
var_resid = decomposition.resid.var()
var_explained = (1 - var_resid / var_total) * 100
print(f"\nVariancia explicada pela decomposicao: {var_explained:.2f}%")

## 8. Deteccao de Outliers

In [ ]:
Q1 = df[target].quantile(0.25)
Q3 = df[target].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iqr = df[(df[target] < lower) | (df[target] > upper)]

from scipy.stats import zscore
z = np.abs(zscore(df[target].dropna()))
outliers_z = df.loc[df.index.isin(df[target].dropna().index[z > 3])]

print("=" * 50)
print("DETECCAO DE OUTLIERS")
print("=" * 50)
print(f"IQR: {len(outliers_iqr)} outliers ({len(outliers_iqr)/len(df)*100:.3f}%)")
print(f"  Limites: [{lower:.0f}, {upper:.0f}] MW")
print(f"Z-score (>3): {len(outliers_z)} outliers ({len(outliers_z)/len(df)*100:.3f}%)")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['timestamp'], df[target], alpha=0.3, linewidth=0.5)
ax.scatter(outliers_iqr['timestamp'], outliers_iqr[target], color='red', s=20, label=f'Outliers IQR ({len(outliers_iqr)})')
ax.axhline(upper, color='red', linestyle='--', alpha=0.5)
ax.axhline(lower, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Data')
ax.set_ylabel('Consumo (MW)')
ax.set_title('Deteccao de Outliers')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Resumo

In [ ]:
print("=" * 70)
print("RESUMO DA ANALISE EXPLORATORIA")
print("=" * 70)
print(f"""
DADOS:
  Periodo: {df['timestamp'].min().date()} a {df['timestamp'].max().date()}
  Registos: {len(df):,}
  Regioes: {', '.join(sorted(df['region'].unique()))}

CONSUMO:
  Media: {df[target].mean():.2f} MW
  Mediana: {df[target].median():.2f} MW
  Desvio-padrao: {df[target].std():.2f} MW
  Amplitude: [{df[target].min():.0f}, {df[target].max():.0f}] MW

PADROES:
  - Forte padrao diario (pico ~14h, vale ~22h)
  - Fim-de-semana com consumo inferior
  - Variacao sazonal (inverno > verao)
  - Lisboa domina o consumo (> 2500 MW medio)

QUALIDADE:
  - Outliers: {len(outliers_iqr)} ({len(outliers_iqr)/len(df)*100:.3f}%)
  - Valores nulos: {df[target].isnull().sum()}
  - Distribuicao: assimetria positiva ({df[target].skew():.3f})
""")